In [10]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import calinski_harabasz_score
# from yellowbrick.cluster import SilhouetteVisualizer
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.base import BaseEstimator, ClusterMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics.cluster import homogeneity_score
from sklearn.metrics.cluster import adjusted_mutual_info_score
import numpy as np
import optuna
from scipy.stats import ks_2samp
from scipy.stats import entropy
from scipy.spatial.distance import jensenshannon
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
import random
import json

In [2]:
def read_data():
    metadata = pd.read_excel('../datasets/metadata_LATINBIOTA_MEXICO.xlsx', sheet_name='Data')
    taxonomy = pd.read_csv('../datasets/bracken_taxonomy.csv', index_col = 0)
    taxonomy = taxonomy.T
    pathways_unstratified = pd.read_csv('../datasets/latinbiota_pathabundance_unstratified.tsv', sep = '\t', index_col = 0)
    pathways_unstratified = pathways_unstratified[[x for x in pathways_unstratified.columns if '.1' not in x]]
    pathways_unstratified.drop('UNMAPPED', axis = 0, inplace = True)
    pathways_unstratified.drop('UNINTEGRATED', axis = 0, inplace = True)
    pathways_unstratified = pathways_unstratified.T
    pathways_unstratified.index = [x.replace('_paired_Abundance', '') for x in pathways_unstratified.index]
    
    return metadata, taxonomy, pathways_unstratified

In [3]:
metadata, taxonomy, pathways = read_data()

In [4]:
sample_outliers = ['37082_3#16','37082_2#4','37035_2#22','37035_7#18','36703_3#20','36703_3#4']

taxonomy.drop(sample_outliers, axis = 0, inplace = True)
pathways.drop(sample_outliers, axis = 0, inplace = True)

In [11]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(taxonomy)

In [23]:
column_variances = list(scaled_data.var(axis=0))

column_variances.sort(reverse=True)
column_variances

[np.float64(0.0654023888285653),
 np.float64(0.06105906416504645),
 np.float64(0.05811019115826463),
 np.float64(0.057784424323989794),
 np.float64(0.05666523572637071),
 np.float64(0.056665235726370694),
 np.float64(0.05493132360225016),
 np.float64(0.054559859599504934),
 np.float64(0.05413035891745926),
 np.float64(0.05407562228097053),
 np.float64(0.05326859430379472),
 np.float64(0.05176299270369884),
 np.float64(0.05175251529795556),
 np.float64(0.051651922023695264),
 np.float64(0.05163285660929212),
 np.float64(0.05161366584385976),
 np.float64(0.051433442184405355),
 np.float64(0.051244949787223486),
 np.float64(0.051200580104965386),
 np.float64(0.05101870971849921),
 np.float64(0.05100012511379463),
 np.float64(0.05094205066459388),
 np.float64(0.05088865171583144),
 np.float64(0.05082386911708721),
 np.float64(0.05057355404180375),
 np.float64(0.05045165862614623),
 np.float64(0.05038716864280012),
 np.float64(0.05028001040912046),
 np.float64(0.05024470331223379),
 np.floa

In [8]:
taxonomy.describe()

,2763670,46228,33039,33038,166486,301301,2763062,39485,418240,2479767,...,1923324,2845935,1475063,35783,103690,2734056,2419609,2704034,3048295,2845556
count,199.000000,1.990000e+02,199.000000,199.000000,1.990000e+02,199.000000,199.000000,1.990000e+02,1.990000e+02,199.000000,...,199.000000,199.000000,199.000000,199.000000,199.000000,199.000000,199.000000,199.000000,199.000000,199.000000
mean,28413.482412,3.613461e+04,87967.326633,17421.618090,8.407536e+04,29515.954774,9265.924623,4.662527e+04,1.425271e+05,21371.251256,...,0.005025,0.005025,0.005025,0.020101,0.175879,0.005025,0.025126,0.040201,0.005025,0.005025
std,62657.008361,1.117680e+05,99446.442465,24023.878232,1.334920e+05,30812.617327,9693.259377,1.009956e+05,1.888418e+05,22067.998397,...,0.070888,0.070888,0.070888,0.283552,2.481084,0.070888,0.354441,0.567105,0.070888,0.070888
min,256.000000,1.113000e+03,3111.000000,1591.000000,3.095000e+03,1177.000000,554.000000,5.570000e+02,6.432000e+03,1501.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3156.000000,5.642500e+03,25125.500000,5861.500000,1.696150e+04,8071.000000,2848.000000,6.955500e+03,3.626100e+04,8198.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6962.000000,9.566000e+03,53159.000000,10149.000000,4.112400e+04,18108.000000,6147.000000,1.686800e+04,6.062100e+04,14103.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,23483.500000,2.263600e+04,120985.000000,19388.500000,9.667950e+04,40896.500000,11188.000000,4.847600e+04,1.459260e+05,26221.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,597259.000000,1.354463e+06,650951.000000,200456.000000,1.000748e+06,206297.000000,52105.000000,1.086294e+06,1.268147e+06,143016.000000,...,1.000000,1.000000,1.000000,4.000000,35.000000,1.000000,5.000000,8.000000,1.000000,1.000000
